In [1]:
from astropy.io import fits
import os
import shutil
from glob import glob
from multiprocessing import Pool
import pandas as pd

In [2]:
Download_directory = '/data/beegfs/astro-storage/groups/beuther/pasteimle/Background_data_ISPY/'
ISPY_directory = '/data/beegfs/astro-storage/groups/beuther/pasteimle/Background_data_ISPY2/'

all_files = glob(os.path.join(Download_directory, 'SPHER*.fits'))
all_files

['/data/beegfs/astro-storage/groups/beuther/pasteimle/Background_data_ISPY/SPHER.2022-04-03T10:18:15.692.fits',
 '/data/beegfs/astro-storage/groups/beuther/pasteimle/Background_data_ISPY/SPHER.2022-04-04T11:06:07.725.fits',
 '/data/beegfs/astro-storage/groups/beuther/pasteimle/Background_data_ISPY/SPHER.2022-04-04T10:33:23.997.fits',
 '/data/beegfs/astro-storage/groups/beuther/pasteimle/Background_data_ISPY/SPHER.2022-04-04T10:24:26.242.fits',
 '/data/beegfs/astro-storage/groups/beuther/pasteimle/Background_data_ISPY/SPHER.2022-04-04T11:27:33.453.fits',
 '/data/beegfs/astro-storage/groups/beuther/pasteimle/Background_data_ISPY/SPHER.2022-04-04T10:23:43.496.fits',
 '/data/beegfs/astro-storage/groups/beuther/pasteimle/Background_data_ISPY/SPHER.2022-04-04T10:24:59.332.fits',
 '/data/beegfs/astro-storage/groups/beuther/pasteimle/Background_data_ISPY/SPHER.2022-04-04T11:06:30.241.fits',
 '/data/beegfs/astro-storage/groups/beuther/pasteimle/Background_data_ISPY/SPHER.2022-04-04T04:41:43.057

In [3]:
def get_info(file):
    header = fits.getheader(file)
    
    try:
        object = header['OBJECT']

        if object.strip() == '':
            return
    
        exp_time = header['EXPTIME']
        obs_date = header['DATE-OBS'].split('T')[0]
        obs_time = header['DATE-OBS'].split('T')[1]

        if int(obs_time[:2]) < 14:
            obs_night = obs_date[:8] + str(int(obs_date[8:11])-1).zfill(2)  
        else: 
            obs_night = obs_date

        filter_name = header['HIERARCH ESO INS COMB IFLT']
        nd_filter = header['HIERARCH ESO INS4 FILT2 NAME']
        type = header['HIERARCH ESO DPR TYPE']

    except KeyError as e:
        print(f"KeyError: {e} in file {file}")
        print(header)
        return

    if type == 'OBJECT':
        type = 'SCIENCE'
    elif 'OBJECT' in type:
        type = type.replace('OBJECT,','')
    if 'DARK,BACKGROUND' in type:
         type = type.replace('DARK,BACKGROUND','DARK')

    if 'DARK,BACKGROUND' in object:
        object = object.replace('DARK,BACKGROUND','DARK')  


    for string in ['DARK,',',LAMP','LAMP,']: #,
        if string in type:
            type = type.replace(string, '')
        if string in object:
            object = object.replace(string, '')

    object = object.replace(' ','')    

    # if object == 'epsSgr':
    #     object = 'HD169022_epsSgr'
    # elif object == 'epsEri':
    #     object = 'HD22049_epsEri'
    # elif object == 'VDEBoo':
    #     object = 'HD131511_VDEBoo'
    # elif object == 'TYC8972-2028-1':
    #     object = 'TYC_8972-2028-1'
    # elif object== 'mu.Tel':
    #     object = 'HD183028_mu.Tel'

    if object == "HIP_36312":
        object = 'HD59659'
    elif object == "TYC_8128_1946_1":
        object = 'HD59659'

    folderpath = os.path.join(object, 'IRDIS')
    folderpath = os.path.join(folderpath, obs_night)

    # if nd_filter != 'OPEN' and type != 'FLUX':
    #     folderpath = os.path.join(folderpath, filter_name.replace('_','_ND'))
    # else:
    folderpath = os.path.join(folderpath, filter_name)

    filepath = os.path.join(folderpath, type + '_' + os.path.basename(file))
    #filepath = os.path.join(folderpath, os.path.basename(file))

    return {
        "orig_file": file,
        "object": object,
        "exp_time": exp_time,
        "obs_night": obs_night,
        "filter_name": filter_name,
        "nd_filter": nd_filter,
        "type": type,
        "filepath": filepath,
        "folderpath": folderpath,
    }

In [4]:
DF = pd.DataFrame()
for file in all_files:
    infos = get_info(file)
    if infos:
        DF = pd.concat([DF, pd.DataFrame([infos])], ignore_index=True)

DF = DF.sort_values(by=["object", "obs_night", "filter_name", "exp_time", "orig_file"], ascending=[True, True, True, True, True]
)

DF = DF[DF['object'] != 'OBJECT']
# DF[DF["object"] == "HIP_36312"] = 'HD59659'
# DF[DF["object"] == "TYC_8128_1946_1"] = 'HD59659'
# DF = DF[DF["object"] != "OBJECT"]
# DF = DF[DF["filter_name"] != "BB_Ks"]
# DF = DF[DF["filter_name"] != "DB_Y23"]

In [5]:
sorted(DF['folderpath'].unique())

['DARK/IRDIS/2022-04-02/DB_Y23',
 'DARK/IRDIS/2022-04-03/DB_K12',
 'DARK/IRDIS/2022-04-03/DB_NDH23',
 'DARK/IRDIS/2022-04-03/DB_Y23',
 'FLAT/IRDIS/2022-04-03/DB_K12',
 'HIP59747/IRDIS/2022-04-03/DB_K12']

In [6]:
for directory in DF['folderpath'].unique():
    full_path = os.path.join(ISPY_directory, directory)
    os.makedirs(full_path, exist_ok=True)

i = 0 
for orig_file, filepath in zip(DF['orig_file'],DF['filepath'].to_list()):
    if i % 100 == 0:
        print(i)
    orig_filepath = os.path.join(orig_file)
    new_filepath = os.path.join(ISPY_directory, filepath)
    shutil.copy2(orig_filepath, new_filepath)
    i += 1

0


In [7]:
unique_objects = DF["object"].unique()
unique_objects

array(['DARK', 'FLAT', 'HIP59747'], dtype=object)

In [ ]:
DF[DF["object"].isin(["HD342", "hd342"])]

In [ ]:
'HD    377'.replace(' ','')

In [ ]:
for file in sorted(glob('/data/beegfs/astro-storage/groups/beuther/pasteimle/Download/*.fits')):
    header = fits.getheader(file)
    object = header['OBJECT']
    print(object, header['HIERARCH ESO DPR TYPE'])

In [21]:
ISPY_directory_new = '/data/beegfs/astro-storage/groups/beuther/pasteimle/ISPY_raw_files/' 
all_files = glob(os.path.join(ISPY_directory_new, '*/*/*/*/*SPHER*.fits'))
for file in all_files:
    print(file, os.path.join(Download_directory, os.path.basename(file)))
    shutil.copy2(file, os.path.join(Download_directory, os.path.basename(file)))

/data/beegfs/astro-storage/groups/beuther/pasteimle/ISPY_raw_files/HD131511_VDEBoo/IRDIS/2023-03-30/DB_H23/CENTER_SPHER.2023-03-31T05:20:36.260.fits /data/beegfs/astro-storage/groups/beuther/pasteimle/Download/CENTER_SPHER.2023-03-31T05:20:36.260.fits
/data/beegfs/astro-storage/groups/beuther/pasteimle/ISPY_raw_files/HD131511_VDEBoo/IRDIS/2023-03-30/DB_H23/SCIENCE_SPHER.2023-03-31T06:42:08.841.fits /data/beegfs/astro-storage/groups/beuther/pasteimle/Download/SCIENCE_SPHER.2023-03-31T06:42:08.841.fits
/data/beegfs/astro-storage/groups/beuther/pasteimle/ISPY_raw_files/HD131511_VDEBoo/IRDIS/2023-03-30/DB_H23/SCIENCE_SPHER.2023-03-31T07:07:14.633.fits /data/beegfs/astro-storage/groups/beuther/pasteimle/Download/SCIENCE_SPHER.2023-03-31T07:07:14.633.fits
/data/beegfs/astro-storage/groups/beuther/pasteimle/ISPY_raw_files/HD131511_VDEBoo/IRDIS/2023-03-30/DB_H23/FLUX_SPHER.2023-03-31T07:19:40.350.fits /data/beegfs/astro-storage/groups/beuther/pasteimle/Download/FLUX_SPHER.2023-03-31T07:19:40.3

In [14]:
for orig_file, filepath in zip(DF['orig_file'],DF['filepath'].to_list()):
    data = fits.getdata(orig_file)

    if data[0].shape == (2048, 2048):
        print(orig_file)
        os.remove(orig_file)

/data/beegfs/astro-storage/groups/beuther/pasteimle/Download 2/SPHER.2015-11-28T12:55:39.633.fits
/data/beegfs/astro-storage/groups/beuther/pasteimle/Download 2/SPHER.2015-11-28T12:34:41.955.fits
/data/beegfs/astro-storage/groups/beuther/pasteimle/Download 2/SPHER.2015-11-28T12:40:47.494.fits
/data/beegfs/astro-storage/groups/beuther/pasteimle/Download 2/SPHER.2015-11-28T12:46:52.624.fits
/data/beegfs/astro-storage/groups/beuther/pasteimle/Download 2/SPHER.2015-11-28T12:56:20.914.fits
/data/beegfs/astro-storage/groups/beuther/pasteimle/Download 2/SPHER.2015-11-28T12:35:33.818.fits
/data/beegfs/astro-storage/groups/beuther/pasteimle/Download 2/SPHER.2015-11-28T12:41:38.814.fits
/data/beegfs/astro-storage/groups/beuther/pasteimle/Download 2/SPHER.2015-11-28T12:48:05.027.fits
/data/beegfs/astro-storage/groups/beuther/pasteimle/Download 2/SPHER.2015-11-29T13:48:39.518.fits
/data/beegfs/astro-storage/groups/beuther/pasteimle/Download 2/SPHER.2015-11-29T13:54:44.391.fits
/data/beegfs/astro-s